# TUES AI Club · От масиви до първи поглед върху данните

**Практика 1 от 2: NumPy, pandas и оригиналният UCI processed Cleveland Heart Disease Dataset.**

Нужни са Python 3, `numpy`, `pandas`, `matplotlib` (в Colab вече са налични). Локално, ако липсват: `python -m pip install numpy pandas matplotlib jupyter` и стартирайте Jupyter в папката на notebook-а. **Изпълнявайте клетките отгоре надолу** (`Run all`); опитайте да предскажете резултата преди всяка клетка. Реалните данни не се променят. Това е учебен набор, не инструмент за медицински решения.

**Маршрут:** измислени масиви → зареждане → DataFrame/Series → качество → target и графики → задачи → решения. Днес **не обучаваме модел**.

## 1. NumPy: малки измислени данни

**Въпрос:** Ако имаме три възрасти, как изглежда един масив? Каква форма очаквате?

NumPy съхранява еднотипни стойности в `ndarray`. `shape` показва размера по оси, `ndim` — броя измерения, `dtype` — типа на стойностите.

In [ ]:
import numpy as np

ages = np.array([42, 55, 61])
print("Масив:", ages)
print("shape:", ages.shape, "ndim:", ages.ndim, "dtype:", ages.dtype)

**След клетката:** Защо формата е `(3,)`, а не `(3, 1)`? **Опитай:** замени първата възраст с друга и изпълни пак.

**Въпрос:** Как да представим 3 човека с по 2 признака? Каква форма очаквате?

Всеки **ред е пример (sample)**; колоните са **признаци (features)**. Тук колона 0 е възраст (години), колона 1 е пулс (уд./мин). Всички числа са **измислени**.

In [ ]:
people = np.array([[42, 160], [55, 130], [61, 145]])
print(people)
print("shape:", people.shape, "ndim:", people.ndim, "dtype:", people.dtype)

**След клетката:** Кое число е пулсът на втория човек? Какво означава `(3, 2)`?

**Въпрос:** Как ще вземем първия ред, първите два реда и всички възрасти?

Индексите започват от 0. В двумерен масив пишем `[редове, колони]`; `:` означава „всички“ по съответната ос, а краят на slice не се включва.

In [ ]:
print("Първи човек:", people[0])
print("Пулс на втория:", people[1, 1])
print("Първи два реда:\n", people[:2, :])
print("Всички възрасти:", people[:, 0])

**След клетката:** Как бихте взели само пулсовете? **Опитай:** промени последния индекс от `0` на `1`.

**Въпрос:** Как ще намерим хората над 50 години?

Сравнението върху цяла колона дава булева маска (`True`/`False`), с която избираме редове.

In [ ]:
over_50 = people[:, 0] > 50
print("Маска:", over_50)
print("Редове над 50 години:\n", people[over_50])

**След клетката:** Колко реда останаха? **Опитай:** смени прага на 60.

**Въпрос:** Средна възраст и среден пулс — по коя ос пресмятаме? А средно за всеки ред?

`axis=0` събира стойности **надолу по редовете** → по един резултат за всяка колона. `axis=1` събира **по колоните** → по един резултат за всеки ред. Втората операция тук смесва години с уд./мин и **няма полезен смисъл** — показва само механиката на осите.

In [ ]:
print("Средни по колони (възраст, пулс):", people.mean(axis=0))
print("Средни по редове (само демонстрация):", people.mean(axis=1))

**След клетката:** Колко числа връща всяка ос и защо? Кое средно бихте използвали в реален анализ?

**Въпрос:** Какво става, ако добавим 1 към Python списък? А към NumPy масив?

Списъкът не извършва аритметика елемент по елемент. NumPy прилага операцията върху всички елементи (векторизация).

In [ ]:
python_list = [42, 55, 61]
print("Списък + [1]:", python_list + [1])  # добавя нов елемент, не прибавя 1 навсякъде
print("Масив + 1:", ages + 1)                # векторизирана операция

**След клетката:** Получихме ли еднакъв брой елементи? **Опитай:** пресметни `ages * 2`.

## 2. Оригиналният Cleveland набор от UCI

**Въпрос:** Откъде идват колоните, ако CSV файлът няма заглавен ред? Как е отбелязана липса?

Четем точно [processed.cleveland.data](https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data), с имената по-долу. `?` се чете като липсваща стойност (`NaN`), **не като 0**. При първо успешно онлайн зареждане записваме оригиналните байтове в `processed.cleveland.data` до notebook-а. След това локалното копие се ползва първо, включително офлайн. В Colab работната папка е временна: за нова офлайн сесия качете и CSV файла от тази папка в `/content` заедно с notebook-а. Ако нямате нито интернет, нито локално копие, е нужна първа онлайн сесия.

In [ ]:
import io
import urllib.request
from pathlib import Path
import pandas as pd

columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]
source_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
cache_path = Path("processed.cleveland.data")

**Въпрос:** Имаме ли вече копие? Ако не — можем ли да го свалим и проверим преди записа?

`pd.read_csv` създава DataFrame. `header=None` пази първия пациент като ред с данни; `na_values="?"` означава липса. Ако изтеглянето се провали, проверете интернет или сложете оригиналния CSV до notebook-а.

In [ ]:
if cache_path.exists():
    df = pd.read_csv(cache_path, header=None, names=columns, na_values="?")
    print("Източник: локален UCI файл:", cache_path.resolve())
else:
    try:
        with urllib.request.urlopen(source_url, timeout=20) as response:
            raw_data = response.read()
        df = pd.read_csv(io.BytesIO(raw_data), header=None, names=columns, na_values="?")
        if df.shape != (303, 14):
            raise ValueError("Неочакван размер; не запазваме този файл.")
        cache_path.write_bytes(raw_data)
        print("Източник: UCI; оригиналът е запазен в", cache_path.resolve())
    except Exception as error:
        raise RuntimeError("Няма валиден локален файл и зареждането от UCI не успя. ") from error
print("Размер (редове, колони):", df.shape)

**След клетката:** Защо проверяваме размера преди да запазим изтегления файл? Ако дойде различна версия, трябва ли да я приемем мълчаливо?

### Речник на 14-те колони

Кодировките са на **този оригинален UCI набор**, не на преработени Kaggle версии. „Без единица“ означава категория или индикатор, не измерена величина.

| Колона | Значение; единица / кодове |
|---|---|
| `age` | възраст, години |
| `sex` | пол: 1 = мъж, 0 = жена; код, без единица |
| `cp` | тип гръдна болка: 1 = типична ангина, 2 = атипична ангина, 3 = неангинозна болка, 4 = безсимптомна; код |
| `trestbps` | кръвно налягане в покой при постъпване, mm Hg |
| `chol` | серумен холестерол, mg/dL |
| `fbs` | кръвна захар на гладно > 120 mg/dL: 1 = да, 0 = не; индикатор |
| `restecg` | ЕКГ в покой: 0 = нормална, 1 = ST–T отклонение, 2 = вероятна/потвърдена левокамерна хипертрофия (критерии на Estes); код |
| `thalach` | достигната максимална сърдечна честота, уд./мин |
| `exang` | ангина, провокирана от натоварване: 1 = да, 0 = не; индикатор |
| `oldpeak` | ST депресия при натоварване спрямо покой; числова стойност (в документацията **не е посочена единица**) |
| `slope` | наклон на ST сегмента при пиково натоварване: 1 = възходящ, 2 = равен, 3 = низходящ; код |
| `ca` | брой големи кръвоносни съдове (0–3), визуализирани чрез флуороскопия; брой |
| `thal` | резултат от изследването: 3 = нормален, 6 = фиксиран дефект, 7 = обратим дефект; код |
| `num` | оригинален етикет (ангиографски статус): 0 = липса, 1–4 = наличие според публикуваното описание; **target**, не входен признак |

Източници: [UCI Heart Disease (страница на набора)](https://archive.ics.uci.edu/dataset/45/heart+disease) и [оригиналният `heart-disease.names` (детайлните значения и кодове)](https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/heart-disease.names). Не правим медицинска интерпретация на кодовете 1–4.

## 3. Първа среща с pandas

**Въпрос:** Как изглеждат първите пациенти, без да отпечатваме цялата таблица?

`df` е **DataFrame** (таблица); един ред е пациент, колоните са признаци и етикет. `head()` дава първите пет реда.

In [ ]:
df.head()

**След клетката:** Коя колона изглежда като етикет, а кои са измервания или категории?

**Въпрос:** Колко реда и колони имаме и как се казват? `shape` и `columns` отговарят на различни въпроси.

In [ ]:
print("Форма:", df.shape)
print("Колони:", df.columns.tolist())

**След клетката:** **Имаме 14 колони. Защо входните features са 13?** Посочете коя колона ще изключим.

**Въпрос:** Кои типове са разчетени и в кои колони има непълни данни? `info()` показва типовете и броя непразни стойности.

In [ ]:
df.info()

**След клетката:** Коя колона има под 303 непразни стойности? Типът `float64` доказва ли, че колоната е непрекъсната величина?

**Въпрос:** Как изглежда една колона, а как — две?

`df["age"]` е **Series** (една колона); `df[["age", "thalach"]]` е DataFrame (списък с имена на колони).

In [ ]:
age_series = df["age"]
print("Една колона:", type(age_series).__name__)
print(age_series.head())
print("Две колони:", type(df[["age", "thalach"]]).__name__)
df[["age", "thalach"]].head()

**След клетката:** Защо при две колони има *двойни* квадратни скоби? **Опитай:** избери `age` и `chol`.

**Въпрос:** Как ще изберем ред по етикет и ред по позиция?

`loc` ползва етикети на редове и имена на колони; `iloc` — числови позиции. В непроменения набор индексът е 0, 1, 2, … и първите редове съвпадат, но това не е гарантирано след филтриране.

In [ ]:
print("По етикет:\n", df.loc[0, ["age", "cp"]])
print("По позиция:\n", df.iloc[0, [0, 2]])

**След клетката:** Съвпадат ли стойностите? Какво ще стане, ако филтрираме редове, без да нулираме индекса?

**Въпрос:** Кои редове имат възраст над 60 години?

Условието създава маска, подобно на NumPy; нищо не изтриваме от `df`.

In [ ]:
older_than_60 = df["age"] > 60
print("Брой:", older_than_60.sum())
df.loc[older_than_60, ["age", "thalach"]].head()

**След клетката:** Показаните пет реда всички ли са над 60? **Опитай:** праг 65.

**Въпрос:** Какви стойности има `cp` и колко пъти се срещат?

`value_counts()` брои категориите. `cp` е записана като число (`float64`), но **по смисъл е категориен признак**: 4 не означава „два пъти повече болка“ от 2.

In [ ]:
df["cp"].value_counts(dropna=False).sort_index()

**След клетката:** Кой код е най-чест? **Опитай:** преброй `exang` по същия начин.

**Въпрос:** Какви са типичните стойности и границите на измерванията?

`describe()` изчислява броя, средната, квартилите, минимума и максимума; избираме измерими величини, **не** кодове като `cp` или етикета `num`.

In [ ]:
df[["age", "trestbps", "chol", "thalach", "oldpeak"]].describe()

**След клетката:** Сравнете броя в `describe()` с броя редове. Може ли числов код на категория да се осреднява смислено?

## 4. Качество: първо проверяваме, после решаваме

**Въпрос:** Къде липсват стойности? `isna()` е маска за липса, `sum()` преброява `True` по колони.

In [ ]:
missing_counts = df.isna().sum()
print(missing_counts)

**След клетката:** Коя колона има най-много липси? Липса на измерване означава ли стойност нула? **Не**: нула е конкретна стойност, а `NaN` означава, че не знаем.

**Въпрос:** Колко голям е проблемът като процент от пациентите? Първо броим, после делим на броя редове.

In [ ]:
missing_percent = missing_counts / len(df) * 100
print(missing_percent.round(2))

**След клетката:** Кои колони бихте разгледали по-внимателно преди следващата лекция?

**Въпрос:** Има ли **точни** повторения на цели редове? `duplicated()` сравнява всички колони, включително `num`; това не е тест за един и същи човек.

In [ ]:
exact_duplicates = df.duplicated()
print("Брой точни повторения:", exact_duplicates.sum())
print("Първи повторения (ако има):")
print(df.loc[exact_duplicates].head())

**След клетката:** Ако намерим съвпадение, как бихме проверили дали е повторен запис или двама различни пациенти? **Не изтриваме автоматично.**

**Въпрос:** Какви са минимумът и максимумът на измерванията? Избираме само количествени колони; кодове като `cp` се проверяват отделно.

In [ ]:
measured_columns = ["age", "trestbps", "chol", "thalach", "oldpeak"]
print(df[measured_columns].agg(["min", "max"]))

**След клетката:** Съмнителна крайна стойност автоматично ли е грешка? За сравнение са нужни документация, контекст и проверка на източника.

**Въпрос:** Има ли непознати кодове сред категориите и колко са чести? Показваме `NaN`, за да не скрием липсите.

In [ ]:
categorical_columns = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal", "num"]
for column in categorical_columns:
    print("\n", column)
    print(df[column].value_counts(dropna=False).sort_index())

**След клетката:** Сравнете кодовете с речника и отбележете липсите. `ca` е брой с малко допустими стойности, затова го проверяваме заедно с категориите.

### Отделен учебен пример — НЕ е от UCI

**Въпрос:** Как изглеждат очевидна грешка, липса и дубликат? Следващата таблица е **измислена**; `df` остава непроменен.

In [ ]:
practice_table = pd.DataFrame({
    "age": [45, 250, 45, None],
    "pulse": [140, 135, 140, 120]
})
print(practice_table)
print("Липси:\n", practice_table.isna().sum())
print("Повторени редове:", practice_table.duplicated().sum())
print("Възрасти > 120:\n", practice_table.loc[practice_table["age"] > 120])

**След клетката:** Какво бихте проверили в първичния запис преди да промените 250 или дублираните редове? Защо не заменяме `NaN` с 0?

## 5. Target и първи графики

**Въпрос:** Кои са оригиналните стойности на `num`? Има ли липсващ етикет? Преди да правим `num > 0`, трябва да знаем дали сравнението няма да скрие липса като `False`.

In [ ]:
print("Оригинални етикети:\n", df["num"].value_counts(dropna=False).sort_index())
missing_labels = df["num"].isna().sum()
print("Липсващи етикети:", missing_labels)
if missing_labels != 0:
    raise ValueError("Има липсващи етикети: спрете и ги прегледайте преди бинаризация.")

**След клетката:** Колко реда са с `num = 0`? Ако имаше липсващи етикети, защо трябваше да спрем?

**Въпрос:** Как да разделим входа от това, което искаме да предскажем?

За тази практика `y = 1`, ако `num > 0`, иначе `0`. `X` съдържа 13 входни признака. **Не включваме `num` в `X`**: така бихме подсказали отговора на бъдещ модел (data leakage). Сега само подготвяме понятията, не обучаваме.

In [ ]:
y = (df["num"] > 0).astype(int)
y.name = "disease_present"
X = df.drop(columns=["num"])
print("X:", X.shape, "y:", y.shape)
print("Има ли num в X?", "num" in X.columns)
print("Класове:\n", y.value_counts().sort_index())

**След клетката:** Обяснете защо 14 колони в `df` дават 13 в `X`. **Опитай:** покажи `X.columns`.

**Въпрос:** Еднакво ли са представени двата класа? Графиката показва брой пациенти в този набор, не честота в населението.

In [ ]:
import matplotlib.pyplot as plt

class_counts = y.value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["0: няма", "1: има"], class_counts.values, color=["steelblue", "coral"])
plt.title("Брой примери по клас (UCI Cleveland)")
plt.xlabel("Двоичен етикет от num")
plt.ylabel("Брой пациенти")
plt.tight_layout()
plt.show()

**Наблюдение:** Кой клас е по-чест и с колко примера? **Интерпретация:** Можем ли от тази извадка да заключим колко често се среща заболяването в цялото население? Не.

**Въпрос:** В кои възрастови интервали са повече от записите? Хистограмата групира стойностите в интервали; височината е брой пациенти.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["age"].dropna(), bins=12, color="steelblue", edgecolor="white")
plt.title("Разпределение на възрастта (UCI Cleveland)")
plt.xlabel("Възраст (години)")
plt.ylabel("Брой пациенти")
plt.tight_layout()
plt.show()

**Наблюдение:** Къде е най-високият стълб и как промяната на `bins` би го променила? **Интерпретация:** Тази извадка представителна ли е за всички възрасти? Не знаем.

**Въпрос:** Как изглеждат възрастта и максималният пулс заедно? Една точка е един запис; графиката сама по себе си не обяснява причините.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["age"], df["thalach"], alpha=0.6, color="teal")
plt.title("Възраст и максимален пулс (UCI Cleveland)")
plt.xlabel("Възраст (години)")
plt.ylabel("Максимален пулс (уд./мин)")
plt.tight_layout()
plt.show()

**Наблюдение:** Има ли разсейване на стойностите при една и съща възраст? **Интерпретация:** Можем ли да правим причинно-следствено или медицинско заключение от точките? Не.

## 6. Упражнения — опитай, преди да отвориш решенията

Изпълни клетките със `TODO` или добави свои под тях; шаблоните не променят данните. Решенията са **само в края**. Ползвай `df`, `X` и `y` от горните клетки.

1. **Основна:** Избери `age` и `chol` и покажи първите **10** реда.
2. **Основна:** Избери записите над **65** години; покажи броя им и първите пет реда.
3. **Основна:** Намери колоната с най-много липсващи стойности и броя им.
4. **Основна:** Направи хистограма на `chol` с надписи и единица mg/dL.
5. **Напреднали:** Чрез `groupby` сравни средната възраст по код на `cp`. Защо това не е медицинско заключение?
6. **Напреднали:** Намери записите с `age > 60` **и** `exang == 1`; колко са? Поставяй всяко условие в скоби.
7. **Напреднали:** Сравни разпределенията на `thalach` за `y == 0` и `y == 1` с две хистограми и легенда. Защо припокриване не означава причинност?

In [ ]:
# TODO 1–3: Напиши своите команди. Например: df[["age", "chol"]].head(10)
# TODO 4: Използвай plt.hist(...), plt.xlabel(...), plt.ylabel(...), plt.show().
# TODO 5–7: Опитай сам/а; решенията са в последната секция.

### Какво остава за лекция 2?

Ще обсъдим **попълване на липси, encoding на категориите и scaling**. Ако по-късно разделим данните на тренировъчни и тестови, параметри като медиана, речник на категории или средна за мащабиране се **научават само от тренировъчните данни** и после се прилагат към тестовите. Иначе получаваме **data leakage**. Днес не правим тези трансформации и не обучаваме модел.

**Финална задача:** „Избери една колона и обясни какво означава, какво трябва да проверим и как би я визуализирал.“

## Решения на упражненията (отвори след самостоятелния опит)

Клетките по-долу могат да се изпълнят подред от чиста сесия след предишните. Сравни своя подход, а не само числата.

**1. Въпрос:** Как избираме две колони и 10 реда? Използваме списък от имена и `head(10)`.

In [ ]:
df[["age", "chol"]].head(10)

**Проверка:** Има ли точно 10 реда и 2 колони?

**2. Въпрос:** Кои пациенти са над 65? Филтърът създава нов изглед, не променя `df`.

In [ ]:
over_65 = df.loc[df["age"] > 65]
print("Брой над 65:", len(over_65))
over_65.head()

**Проверка:** Всички показани възрасти ли са строго над 65?

**3. Въпрос:** Коя колона има най-много липси? `idxmax` връща името на първия максимум.

In [ ]:
missing_by_column = df.isna().sum()
most_missing_column = missing_by_column.idxmax()
print("Колона:", most_missing_column)
print("Брой липси:", missing_by_column.max())

**Проверка:** Ако всички стойности бяха 0, щеше ли `idxmax` все пак да върне име?

**4. Въпрос:** Как се разпределя холестеролът? Липсите се пропускат само за графиката.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["chol"].dropna(), bins=12, edgecolor="white", color="darkorange")
plt.title("Разпределение на холестерола (UCI Cleveland)")
plt.xlabel("Серумен холестерол (mg/dL)")
plt.ylabel("Брой пациенти")
plt.tight_layout()
plt.show()

**Проверка:** Ако сменим броя интервали, как се променя формата? Не правим медицински извод.

**5. Въпрос:** Как се различава средната възраст между кодовете на `cp`? Групираме по категория.

In [ ]:
mean_age_by_cp = df.groupby("cp")["age"].mean()
print(mean_age_by_cp)

**Проверка:** Може ли от наблюдавана разлика в тази извадка да твърдим причинност? Не.

**6. Въпрос:** Кои редове изпълняват едновременно двете условия? `&` комбинира маските.

In [ ]:
both_conditions = (df["age"] > 60) & (df["exang"] == 1)
print("Брой:", both_conditions.sum())
print(df.loc[both_conditions, ["age", "exang"]].head())

**Проверка:** Защо използваме `&`, а не Python `and` за Series?

**7. Въпрос:** Припокриват ли се разпределенията на максималния пулс по клас? Ползваме еднакви интервали.

In [ ]:
shared_bins = np.arange(60, 211, 10)
plt.figure(figsize=(7, 4))
plt.hist(df.loc[y == 0, "thalach"].dropna(), bins=shared_bins, alpha=0.5, label="Клас 0")
plt.hist(df.loc[y == 1, "thalach"].dropna(), bins=shared_bins, alpha=0.5, label="Клас 1")
plt.title("Максимален пулс по двоичен клас (UCI Cleveland)")
plt.xlabel("Максимален пулс (уд./мин)")
plt.ylabel("Брой пациенти")
plt.legend(title="Етикет от num")
plt.tight_layout()
plt.show()

**Проверка:** Как би изглеждало сравнението при различен брой пациенти в класовете? Графиката показва асоциации, не причинност или диагноза.